# Task multi-label training

Trains one or more `(model, multiplier)` pairs on the same target species. Each pair writes its own `model.keras`, `train.json`, `model.tflite`, `results.json`, `results.npz`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os

import pyrootutils

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

## Parameters

In [ ]:
# On-disk dataset folder from 01_dataset_build.ipynb.
COLLECTION = "task_s2_49_2_r50"

from building.models import BASE_LIST
from building.polychirp_training.training import TinyChirpModel, TrainHP
MODELS = BASE_LIST + [  # curated set plus notebook extras
    TinyChirpModel(base_model="squeezenet_mel", multi=1.0),
    TinyChirpModel(base_model="squeezenet_time", multi=1.0),
    TinyChirpModel(base_model="mel_cnn", multi=2.0),
]

HP = TrainHP(
    epochs=1, 
    patience=10, min_delta=1e-3,
    batch_size=32, seed=42,
    lr_patience=5, lr_factor=0.5, lr_min=1e-5,
    augment=True,
)

## Load catalog

Loaded once. `train_all` builds one raw-audio dataset and featurizes per unique `input_repr`, so every spec sees the same clip order.

In [ ]:
import logging
import warnings

import tensorflow as tf

tf.get_logger().setLevel("ERROR")
logging.getLogger("absl").setLevel("ERROR")
warnings.filterwarnings("ignore", message=".*AudioResample.*")

from building.polychirp_training.training import load_dataset_catalog

catalog = load_dataset_catalog(COLLECTION)
target_names = [n for n in catalog.class_names if n != "non_target"]
print(f"target species ({len(target_names)}): {target_names}")
print(f"non_target idx : {catalog.class_names.index('non_target')}")

## Co-train every spec on shared batches


In [ ]:
from building.polychirp_training.results import train_all, MultiModelLivePlot, spec_label

specs  = list(MODELS)
labels = [spec_label(s) for s in specs]
live   = MultiModelLivePlot(labels)

def _on_epoch_end(spec, epoch, logs):
    live.update(spec_label(spec), logs)

results = train_all(specs, catalog, COLLECTION, hp=HP, on_epoch_end=_on_epoch_end)